# Bài 3: PCA - Principal Component Analysis

**Session 7 - Advanced Data Science with Python**

---

## Mục tiêu bài học

- Hiểu **PCA làm gì** và **tại sao cần** giảm chiều
- Nắm vững **eigenvalue, eigenvector, explained variance** nghĩa là gì
- Biết cách **chọn số components tối ưu**
- Ứng dụng PCA vào **tiền xử lý** và **visualization**
- Hiểu **loadings** để diễn giải PCA

---

## 1. Tại Sao Cần Giảm Chiều?

### Curse of Dimensionality (Lời nguyền chiều cao)

Khi số features tăng:
- Khoảng cách giữa các điểm **trở nên gần bằng nhau** → KNN, K-Means thất bại
- Cần **lượng data tăng theo hàm mũ** để bao phủ không gian
- **Overfitting** dễ xảy ra hơn

### PCA giải quyết bằng cách:
- Tìm **hướng (directions)** mà dữ liệu **biến thiên nhiều nhất**
- Chiếu dữ liệu lên các hướng đó → giảm features mà **giữ lại thông tin quan trọng**

---

## 2. PCA Hoạt Động Như Thế Nào?

### Các bước:

```
1. Chuẩn hóa dữ liệu (mean=0, std=1)
2. Tính Covariance Matrix: C = (1/n) * X^T * X
3. Tìm Eigenvalues & Eigenvectors của C
4. Sắp xếp eigenvectors theo eigenvalue GIẢM DẦN
5. Chọn top-k eigenvectors → ma trận chiếu W
6. Chiếu: X_new = X * W (giảm từ d features → k features)
```

### Ý nghĩa:

| Khái niệm | Ý nghĩa |
|:---|:---|
| **Eigenvector** | HƯỚNG mà dữ liệu biến thiên → Principal Component (PC) |
| **Eigenvalue** | ĐỘ LỚN biến thiên theo hướng đó |
| **Explained Variance Ratio** | % thông tin giữ lại: $\frac{\lambda_k}{\sum \lambda}$ |
| **PC1** | Hướng biến thiên NHIỀU NHẤT |
| **PC2** | Hướng biến thiên nhiều THỨ 2 (vuông góc PC1) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris, load_digits, load_wine
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
print("✅ Thư viện sẵn sàng!")

In [ ]:
# Demo trực quan: PCA trên dữ liệu 2D → 1D
np.random.seed(42)
# Tạo dữ liệu tương quan mạnh
x = np.random.randn(200)
y = 0.8 * x + 0.3 * np.random.randn(200)
X_2d = np.column_stack([x, y])
X_2d_scaled = StandardScaler().fit_transform(X_2d)

pca = PCA(2)
pca.fit(X_2d_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vẽ dữ liệu + PC directions
axes[0].scatter(X_2d_scaled[:,0], X_2d_scaled[:,1], c='steelblue', s=20, alpha=0.5)

origin = [0, 0]
for i, (comp, var, color) in enumerate(zip(pca.components_, pca.explained_variance_, ['red', 'green'])):
    scale = np.sqrt(var) * 3
    axes[0].annotate('', xy=(comp[0]*scale, comp[1]*scale), xytext=origin,
                     arrowprops=dict(arrowstyle='->', color=color, lw=3))
    axes[0].text(comp[0]*scale*1.1, comp[1]*scale*1.1, 
                 f'PC{i+1} ({pca.explained_variance_ratio_[i]*100:.0f}%)',
                 color=color, fontsize=12, fontweight='bold')
axes[0].set_title('PCA tìm 2 hướng biến thiên chính', fontweight='bold')
axes[0].set_aspect('equal'); axes[0].grid(alpha=0.3)

# Chiếu xuống PC1 (1D)
X_projected = pca.transform(X_2d_scaled)
axes[1].scatter(X_projected[:, 0], np.zeros_like(X_projected[:,0]), c='steelblue', s=20, alpha=0.5)
axes[1].set_title(f'Chiếu xuống PC1 → Giữ {pca.explained_variance_ratio_[0]*100:.0f}% thông tin', 
                   fontweight='bold')
axes[1].set_xlabel('PC1')
axes[1].set_yticks([])

plt.tight_layout(); plt.show()
print(f"2D → 1D nhưng giữ lại {pca.explained_variance_ratio_[0]*100:.0f}% variance!")

---

## 3. Chọn Số Components (n_components)

### Quy tắc:
- **Cumulative Explained Variance ≥ 85-95%**: giữ đủ thông tin
- Vẽ **Scree Plot** → tìm điểm "khuỷu tay" (giống Elbow trong K-Means)
- Có thể dùng `n_components = 0.95` → sklearn tự chọn!

In [ ]:
# Demo: Chọn n_components trên Digits dataset (64 features → ???)
digits = load_digits()
X_digits = StandardScaler().fit_transform(digits.data)

pca_full = PCA().fit(X_digits)  # Fit tất cả 64 components

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, 21), pca_full.explained_variance_ratio_[:20], color='steelblue', alpha=0.8)
axes[0].set_xlabel('Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot (top 20 components)', fontweight='bold')

# Cumulative
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, 65), cum_var, 'bo-', ms=3)
axes[1].axhline(y=0.90, color='red', ls='--', label='90%')
axes[1].axhline(y=0.95, color='orange', ls='--', label='95%')

# Tìm n tối thiểu cho 90% và 95%
n_90 = np.argmax(cum_var >= 0.90) + 1
n_95 = np.argmax(cum_var >= 0.95) + 1
axes[1].axvline(x=n_90, color='red', ls=':', alpha=0.5)
axes[1].axvline(x=n_95, color='orange', ls=':', alpha=0.5)
axes[1].set_xlabel('Components'); axes[1].set_ylabel('Cumulative Variance')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"64 features → {n_90} components giữ 90% | {n_95} components giữ 95%")
print(f"→ Giảm {64-n_90} features mà chỉ mất 10% thông tin!")

In [ ]:
# Trick: n_components = 0.95 → tự chọn
pca_auto = PCA(n_components=0.95)
X_reduced = pca_auto.fit_transform(X_digits)
print(f"sklearn tự chọn {pca_auto.n_components_} components để giữ 95% variance")
print(f"Shape: {X_digits.shape} → {X_reduced.shape}")

---

## 4. Ứng Dụng 1: Visualization (PCA 2D/3D)

Dữ liệu nhiều chiều → **không thể vẽ trực tiếp**. PCA giúp chiếu xuống 2D/3D.

In [ ]:
# Digits: 64D → 2D visualization
X_vis = PCA(2).fit_transform(X_digits)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_vis[:,0], X_vis[:,1], c=digits.target, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, label='Digit')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Digits Dataset: 64D → 2D bằng PCA\n(mỗi màu = 1 chữ số)', fontsize=13, fontweight='bold')
plt.grid(alpha=0.3); plt.show()

print("Các chữ số cùng loại tụ gần nhau → PCA bảo toàn cấu trúc tốt!")

In [ ]:
# Iris: 4D → 2D
iris = load_iris()
X_iris_s = StandardScaler().fit_transform(iris.data)
X_iris_pca = PCA(2).fit_transform(X_iris_s)

plt.figure(figsize=(8, 6))
for i, name in enumerate(iris.target_names):
    mask = iris.target == i
    plt.scatter(X_iris_pca[mask, 0], X_iris_pca[mask, 1], label=name, s=50, alpha=0.7)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Iris: 4D → 2D bằng PCA', fontsize=13, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3); plt.show()

---

## 5. Ứng Dụng 2: PCA + Clustering (Pipeline)

PCA giảm chiều → K-Means clustering **trên không gian mới** → kết quả tốt hơn!

In [ ]:
# Digits: Clustering trước và sau PCA
from sklearn.pipeline import Pipeline

# Không PCA
labels_raw = KMeans(10, random_state=42, n_init=10).fit_predict(X_digits)
sil_raw = silhouette_score(X_digits, labels_raw)

# Có PCA (giữ 95%)
X_pca = PCA(n_components=0.95).fit_transform(X_digits)
labels_pca = KMeans(10, random_state=42, n_init=10).fit_predict(X_pca)
sil_pca = silhouette_score(X_pca, labels_pca)

print(f"KHÔNG PCA (64D): Silhouette = {sil_raw:.3f}")
print(f"CÓ PCA ({X_pca.shape[1]}D): Silhouette = {sil_pca:.3f}")
print(f"\n→ PCA giảm noise → Clustering TỐT HƠN!")

---

## 6. Loadings: Diễn Giải PCA

**Loadings** = trọng số của mỗi feature gốc trong từng PC.

→ Biết **feature nào đóng góp nhiều nhất** vào mỗi component.

In [ ]:
# Demo loadings trên Wine dataset
wine = load_wine()
X_wine = StandardScaler().fit_transform(wine.data)
pca_w = PCA(n_components=3).fit(X_wine)

# Loadings = components_ (mỗi hàng là 1 PC, mỗi cột là 1 feature)
loadings = pd.DataFrame(pca_w.components_.T, 
                         index=wine.feature_names, 
                         columns=[f'PC{i+1}' for i in range(3)])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap loadings
sns.heatmap(loadings, annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=axes[0])
axes[0].set_title('PCA Loadings\n(feature nào ảnh hưởng PC nào)', fontweight='bold')

# Biplot
X_wine_pca = pca_w.transform(X_wine)
axes[1].scatter(X_wine_pca[:,0], X_wine_pca[:,1], c=wine.target, cmap='Set1', s=20, alpha=0.5)

for i, name in enumerate(wine.feature_names):
    axes[1].annotate('', xy=(loadings.iloc[i, 0]*5, loadings.iloc[i, 1]*5), xytext=(0,0),
                     arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
    axes[1].text(loadings.iloc[i, 0]*5.2, loadings.iloc[i, 1]*5.2, name, fontsize=7, color='red')

axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('Biplot: Dữ liệu + Feature directions', fontweight='bold')

plt.tight_layout(); plt.show()
print("→ Mũi tên đỏ = hướng feature. Dài = đóng góp nhiều")
print("→ Features cùng hướng = tương quan dương")

---

## 7. PCA vs Các Phương Pháp Giảm Chiều Khác

| Phương pháp | Loại | Bảo toàn | Khi nào dùng |
|:---|:---|:---|:---|
| **PCA** | Tuyến tính | Variance (global) | Tiền xử lý, feature extraction |
| **t-SNE** | Phi tuyến | Local structure | Visualization 2D/3D |
| **UMAP** | Phi tuyến | Local + Global | Visualization (nhanh hơn t-SNE) |
| **ICA** | Tuyến tính | Statistical independence | Tách nguồn tín hiệu |

In [ ]:
# So sánh PCA vs t-SNE trên Digits
from sklearn.manifold import TSNE

X_tsne = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_digits)
X_pca_2d = PCA(2).fit_transform(X_digits)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, X_plot, title in [
    (axes[0], X_pca_2d, 'PCA (tuyến tính, nhanh)'),
    (axes[1], X_tsne, 't-SNE (phi tuyến, chậm hơn)')
]:
    sc = ax.scatter(X_plot[:,0], X_plot[:,1], c=digits.target, cmap='tab10', s=10, alpha=0.7)
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.colorbar(sc, ax=ax)

plt.suptitle('PCA vs t-SNE: Vis 64D Digits', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print("📌 t-SNE tách các cụm rõ hơn (phi tuyến)")
print("📌 Nhưng t-SNE CHẬM + không thể transform dữ liệu mới")
print("📌 PCA: nhanh, transform được, phù hợp pipeline ML")

---

## 8. Bài Thực Hành: PCA trên Wholesale Customers

**Yêu cầu**:
1. Load Wholesale Customers + Scale
2. PCA full → Scree plot + Cumulative variance
3. Chọn n_components giữ 90%
4. Phân tích Loadings: Feature nào quan trọng nhất?
5. PCA 2D + K-Means clustering + Visualization

In [ ]:
# =============================================
# GỢI Ý BÀI GIẢI
# =============================================
df = pd.read_csv('../Bai thi thu/Course Files/Wholesale customers data.csv')
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

# 1. Scale
X_ws = StandardScaler().fit_transform(np.log1p(df[features]))

# 2. PCA full
pca_full = PCA().fit(X_ws)
cum = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 7), pca_full.explained_variance_ratio_, alpha=0.6, label='Individual')
ax.plot(range(1, 7), cum, 'ro-', label='Cumulative')
ax.axhline(y=0.9, color='green', ls='--', label='90%')
ax.set_xlabel('Component'); ax.set_ylabel('Variance')
ax.set_title('Scree Plot - Wholesale Customers', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

n_90 = np.argmax(cum >= 0.9) + 1
print(f"Cần {n_90} components để giữ 90% variance")

# 3. Loadings
loadings = pd.DataFrame(pca_full.components_[:3].T, index=features, 
                         columns=['PC1', 'PC2', 'PC3'])
print(f"\nLoadings:")
print(loadings.round(3).to_string())

# 4. PCA 2D + K-Means
X_pca = PCA(2).fit_transform(X_ws)
labels = KMeans(3, random_state=42, n_init=10).fit_predict(X_ws)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:,0], X_pca[:,1], c=labels, cmap='Set1', s=40, alpha=0.7)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Wholesale: PCA 2D + K-Means', fontweight='bold')
plt.colorbar(scatter, label='Cluster'); plt.grid(alpha=0.3)
plt.show()

---

## 📌 TỔNG HỢP: 20% Kiến Thức → 80% Ứng Dụng

| # | Kiến thức cốt lõi | Chi tiết |
|:--|:---|:---|
| 1 | **PCA = Tìm hướng variance max** | Eigenvectors = hướng, Eigenvalues = độ lớn |
| 2 | **Chọn n qua Cumulative Variance** | ≥ 90-95%. Hoặc `PCA(n_components=0.95)` |
| 3 | **Loadings = diễn giải** | `pca.components_` → feature nào đóng góp PC nào |
| 4 | **PCA áp dụng TRƯỚC clustering** | Giảm noise, giảm chiều → clustering tốt hơn |
| 5 | **PCA tuyến tính, t-SNE phi tuyến** | Visualization: t-SNE đẹp hơn. Pipeline ML: PCA |

### Code Template:
```python
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Luôn scale trước!
X_scaled = StandardScaler().fit_transform(X)

# PCA giữ 95% variance
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_scaled)
print(f'Giữ {pca.n_components_} components, {sum(pca.explained_variance_ratio_)*100:.1f}% variance')
```

---
**→ Bài tiếp theo: [Bài 4] ICA - Independent Component Analysis**

# Bài 3: PCA - Principal Component Analysis Từ Scratch

**Session 7 - Advanced Data Science with Python**

---

## Mục tiêu bài học

Sau khi hoàn thành bài này, bạn sẽ:
- Hiểu **bản chất toán học** của PCA (Eigenvalues, Eigenvectors)
- **Code PCA từ scratch** bằng NumPy
- Biết cách chọn **số thành phần chính (components)**
- Áp dụng PCA cho **trực quan hóa** và **giảm chiều dữ liệu**
- Hiểu **khi nào nên và không nên** dùng PCA

---

## 1. PCA Là Gì? - Trực Giác

### 1.1 Ý tưởng cốt lõi

**PCA (Principal Component Analysis)** tìm các **trục mới** (principal components) sao cho:
- Trục thứ nhất (PC1) nắm bắt **nhiều phương sai nhất** trong dữ liệu
- Trục thứ hai (PC2) nắm bắt nhiều phương sai nhất **còn lại** (vuông góc với PC1)
- Cứ tiếp tục...

### 1.2 Ví dụ trực giác

Hãy tưởng tượng bạn chụp ảnh một con mèo 3D:
- Ảnh chụp từ **phía trước** → thấy rõ nhất (= PC1)
- Ảnh chụp từ **phía bên** → thấy thêm thông tin mới (= PC2)
- Ảnh chụp từ **trên xuống** → ít thông tin mới (= PC3, có thể bỏ)

→ PCA tìm **góc chụp tốt nhất** để giữ lại nhiều thông tin nhất!

### 1.3 Phương sai = Thông tin

Trong PCA, **phương sai (variance) = thông tin**:
- Feature có phương sai cao → chứa nhiều thông tin hữu ích
- Feature có phương sai thấp → gần như hằng số → ít giá trị
- PCA tìm hướng (direction) có phương sai LỚN NHẤT

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# Tạo dữ liệu 2D có tương quan
np.random.seed(42)
n = 200

# Dữ liệu có tương quan mạnh: y ≈ 2x + noise
x = np.random.randn(n) * 2
y = 2 * x + np.random.randn(n) * 1
X_2d = np.column_stack([x, y])

# Trung tâm hóa (center)
X_centered = X_2d - X_2d.mean(axis=0)

# Tính PCA thủ công
cov_matrix = np.cov(X_centered.T)
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Sắp xếp giảm dần
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Trực quan
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Dữ liệu gốc với PCA directions
axes[0].scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.5, s=30, c='steelblue')

# Vẽ PC1 và PC2
origin = [0, 0]
scale = 3
colors_pc = ['red', 'green']
for i, (val, vec) in enumerate(zip(eigenvalues, eigenvectors.T)):
    axes[0].annotate('', xy=vec*scale*np.sqrt(val), xytext=origin,
                     arrowprops=dict(arrowstyle='->', color=colors_pc[i], lw=3))
    axes[0].text(vec[0]*scale*np.sqrt(val)+0.3, vec[1]*scale*np.sqrt(val)+0.3, 
                 f'PC{i+1}\n(var={val:.1f})', fontsize=12, color=colors_pc[i], fontweight='bold')

axes[0].set_xlabel('X1', fontsize=12)
axes[0].set_ylabel('X2', fontsize=12)
axes[0].set_title('Dữ liệu gốc + Hướng PCA\n(Mũi tên = Principal Components)', fontsize=13, fontweight='bold')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Dữ liệu sau khi chiếu lên PC1, PC2
X_pca = X_centered @ eigenvectors
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=30, c='steelblue')
axes[1].set_xlabel(f'PC1 ({eigenvalues[0]/eigenvalues.sum()*100:.1f}% variance)', fontsize=12)
axes[1].set_ylabel(f'PC2 ({eigenvalues[1]/eigenvalues.sum()*100:.1f}% variance)', fontsize=12)
axes[1].set_title('Dữ liệu sau PCA\n(Đã xoay theo trục mới)', fontsize=13, fontweight='bold')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Explained Variance Ratio:")
print(f"  PC1: {eigenvalues[0]/eigenvalues.sum()*100:.1f}% (hướng có phương sai lớn nhất)")
print(f"  PC2: {eigenvalues[1]/eigenvalues.sum()*100:.1f}% (hướng vuông góc)")
print(f"\n→ Chỉ cần PC1 đã giữ được ~{eigenvalues[0]/eigenvalues.sum()*100:.0f}% thông tin!")

---

## 2. Nền Tảng Toán Học

### 2.1 Các bước toán học của PCA

Cho ma trận dữ liệu $X$ kích thước $(n \times d)$ ($n$ mẫu, $d$ features):

**Bước 1: Trung tâm hóa (Centering)**
$$\bar{X} = X - \mu$$
Trong đó $\mu$ là vector trung bình của mỗi cột.

**Bước 2: Tính Ma trận Hiệp phương sai (Covariance Matrix)**
$$C = \frac{1}{n-1} \bar{X}^T \bar{X}$$
Ma trận $C$ kích thước $(d \times d)$, đo mức độ biến thiên cùng nhau của các features.

**Bước 3: Phân tích giá trị riêng (Eigendecomposition)**
$$C \cdot v_i = \lambda_i \cdot v_i$$
- $\lambda_i$: **giá trị riêng (eigenvalue)** = phương sai theo hướng $v_i$
- $v_i$: **vector riêng (eigenvector)** = hướng principal component

**Bước 4: Sắp xếp theo eigenvalue giảm dần**
$$\lambda_1 \geq \lambda_2 \geq ... \geq \lambda_d$$

**Bước 5: Chọn k components và chiếu (project)**
$$X_{\text{new}} = \bar{X} \cdot W_k$$
Trong đó $W_k$ là ma trận chứa $k$ eigenvectors đầu tiên.

### 2.2 Explained Variance Ratio

$$\text{EVR}_i = \frac{\lambda_i}{\sum_{j=1}^{d}\lambda_j}$$

Cho biết % thông tin mà component thứ $i$ giữ lại.

In [ ]:
# Minh họa từng bước toán học chi tiết

# Dữ liệu mẫu nhỏ để dễ theo dõi
np.random.seed(42)
X_small = np.array([
    [2.5, 2.4],
    [0.5, 0.7],
    [2.2, 2.9],
    [1.9, 2.2],
    [3.1, 3.0],
    [2.3, 2.7],
    [2.0, 1.6],
    [1.0, 1.1],
    [1.5, 1.6],
    [1.1, 0.9],
])

print("=" * 60)
print("BƯỚC 1: Trung tâm hóa")
print("=" * 60)
mean = X_small.mean(axis=0)
X_centered = X_small - mean
print(f"Trung bình: {mean}")
print(f"\nDữ liệu gốc (5 dòng đầu):\n{X_small[:5]}")
print(f"\nSau trung tâm hóa (5 dòng đầu):\n{X_centered[:5].round(4)}")

print("\n" + "=" * 60)
print("BƯỚC 2: Ma trận Hiệp phương sai")
print("=" * 60)
cov_matrix = np.cov(X_centered.T)  # equivalent to (X_centered.T @ X_centered) / (n-1)
print(f"\nCov Matrix (2x2):")
print(f"  [{cov_matrix[0,0]:.4f}  {cov_matrix[0,1]:.4f}]")
print(f"  [{cov_matrix[1,0]:.4f}  {cov_matrix[1,1]:.4f}]")
print(f"\nDiagonal = variance của từng feature")
print(f"Off-diagonal = covariance giữa 2 features (tương quan mạnh = giá trị lớn)")

print("\n" + "=" * 60)
print("BƯỚC 3: Eigen Decomposition")
print("=" * 60)
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
# Sắp xếp giảm dần
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print(f"\nEigenvalues (phương sai theo mỗi hướng):")
for i, val in enumerate(eigenvalues):
    print(f"  λ{i+1} = {val:.4f}")

print(f"\nEigenvectors (hướng principal components):")
for i in range(len(eigenvalues)):
    print(f"  v{i+1} = {eigenvectors[:, i].round(4)}")

print("\n" + "=" * 60)
print("BƯỚC 4 & 5: Chiếu dữ liệu lên trục mới")
print("=" * 60)
evr = eigenvalues / eigenvalues.sum()
print(f"\nExplained Variance Ratio:")
print(f"  PC1: {evr[0]*100:.2f}%")
print(f"  PC2: {evr[1]*100:.2f}%")

# Chiếu lên PC1 (giảm từ 2D → 1D)
X_projected_1d = X_centered @ eigenvectors[:, 0:1]
print(f"\nDữ liệu gốc: 2D → Sau PCA (1 component): 1D")
print(f"Shape: {X_small.shape} → {X_projected_1d.shape}")
print(f"Giữ lại {evr[0]*100:.2f}% thông tin!")

---

## 3. PCA Từ Scratch - Class Hoàn Chỉnh

In [ ]:
class PCAFromScratch:
    """
    PCA triển khai từ đầu bằng NumPy.
    
    Parameters:
    -----------
    n_components : int or float
        - Nếu int: số components muốn giữ
        - Nếu float (0-1): giữ đủ components để đạt tỷ lệ variance này
    """
    
    def __init__(self, n_components=None):
        self.n_components = n_components
    
    def fit(self, X):
        """Tính các principal components"""
        n_samples, n_features = X.shape
        
        # Bước 1: Trung tâm hóa
        self.mean_ = X.mean(axis=0)
        X_centered = X - self.mean_
        
        # Bước 2: Ma trận hiệp phương sai
        cov_matrix = np.cov(X_centered.T)
        
        # Bước 3: Eigen decomposition
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # Bước 4: Sắp xếp giảm dần
        idx = np.argsort(eigenvalues)[::-1]
        self.eigenvalues_ = eigenvalues[idx]
        self.eigenvectors_ = eigenvectors[:, idx]
        
        # Explained variance ratio
        total_var = self.eigenvalues_.sum()
        self.explained_variance_ratio_ = self.eigenvalues_ / total_var
        self.cumulative_variance_ratio_ = np.cumsum(self.explained_variance_ratio_)
        
        # Xác định số components
        if self.n_components is None:
            self.n_components_ = n_features
        elif isinstance(self.n_components, float) and 0 < self.n_components < 1:
            # Tìm số components nhỏ nhất đạt ngưỡng variance
            self.n_components_ = np.searchsorted(self.cumulative_variance_ratio_, self.n_components) + 1
        else:
            self.n_components_ = int(self.n_components)
        
        # Ma trận chiếu
        self.components_ = self.eigenvectors_[:, :self.n_components_].T  # (k, d)
        
        return self
    
    def transform(self, X):
        """Chiếu dữ liệu lên các principal components"""
        X_centered = X - self.mean_
        return X_centered @ self.components_.T
    
    def fit_transform(self, X):
        """Fit và transform"""
        self.fit(X)
        return self.transform(X)
    
    def inverse_transform(self, X_transformed):
        """Khôi phục dữ liệu gốc (xấp xỉ)"""
        return X_transformed @ self.components_ + self.mean_

print("✅ Class PCAFromScratch đã được định nghĩa!")

In [ ]:
# Test PCA từ scratch vs sklearn
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

# Load Iris dataset (4 features)
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Chuẩn hóa
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)

# PCA từ scratch
pca_scratch = PCAFromScratch(n_components=2)
X_scratch = pca_scratch.fit_transform(X_scaled)

# PCA sklearn
pca_sklearn = SklearnPCA(n_components=2)
X_sklearn = pca_sklearn.fit_transform(X_scaled)

# So sánh
print("=" * 60)
print("SO SÁNH PCA từ Scratch vs sklearn")
print("=" * 60)
print(f"\nExplained Variance Ratio:")
print(f"  Scratch: {pca_scratch.explained_variance_ratio_[:2].round(6)}")
print(f"  Sklearn: {pca_sklearn.explained_variance_ratio_.round(6)}")
print(f"\nTổng variance giữ lại:")
print(f"  Scratch: {pca_scratch.explained_variance_ratio_[:2].sum()*100:.2f}%")
print(f"  Sklearn: {pca_sklearn.explained_variance_ratio_.sum()*100:.2f}%")

# Trực quan hóa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
target_names = iris.target_names
colors = ['#e74c3c', '#3498db', '#2ecc71']

for ax, X_proj, title in zip(axes, [X_scratch, X_sklearn], ['PCA từ Scratch', 'PCA sklearn']):
    for i, (name, color) in enumerate(zip(target_names, colors)):
        mask = y_iris == i
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1], c=color, label=name, s=40, alpha=0.7)
    ax.set_xlabel('PC1', fontsize=12)
    ax.set_ylabel('PC2', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\n✅ Kết quả gần giống nhau! (Dấu +/- có thể khác do convention)")

---

## 4. Chọn Số Components Tối Ưu

### Phương pháp 1: Cumulative Explained Variance ≥ 95%

Giữ đủ components để tổng variance giải thích được ≥ 95% (hoặc 90%).

### Phương pháp 2: Scree Plot

Tìm điểm "khuỷu tay" trong đồ thị eigenvalue vs component.

In [ ]:
# Demo: Chọn số components cho Iris dataset
pca_full = PCAFromScratch(n_components=None)
pca_full.fit(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_features = len(pca_full.explained_variance_ratio_)

# Scree Plot
axes[0].bar(range(1, n_features+1), pca_full.explained_variance_ratio_, 
            alpha=0.7, color='steelblue', label='Individual')
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Explained Variance Ratio', fontsize=12)
axes[0].set_title('Scree Plot', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(1, n_features+1))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for i, v in enumerate(pca_full.explained_variance_ratio_):
    axes[0].text(i+1, v + 0.01, f'{v*100:.1f}%', ha='center', fontsize=10)

# Cumulative Variance
axes[1].plot(range(1, n_features+1), pca_full.cumulative_variance_ratio_, 
             'ro-', linewidth=2, markersize=8)
axes[1].axhline(y=0.95, color='green', linestyle='--', label='Ngưỡng 95%')
axes[1].axhline(y=0.90, color='orange', linestyle='--', label='Ngưỡng 90%')
axes[1].set_xlabel('Số Components', fontsize=12)
axes[1].set_ylabel('Cumulative Explained Variance', fontsize=12)
axes[1].set_title('Cumulative Variance', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(1, n_features+1))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

for i, v in enumerate(pca_full.cumulative_variance_ratio_):
    axes[1].text(i+1.1, v - 0.03, f'{v*100:.1f}%', fontsize=10)

plt.tight_layout()
plt.show()

# Auto-select
pca_auto = PCAFromScratch(n_components=0.95)
pca_auto.fit(X_scaled)
print(f"\n📊 Để giữ 95% variance: cần {pca_auto.n_components_} components (từ 4 features ban đầu)")
print(f"→ Giảm từ 4D → {pca_auto.n_components_}D mà chỉ mất ~{(1-0.95)*100:.0f}% thông tin!")

---

## 5. Ứng Dụng 1: Trực Quan Hóa Dữ Liệu Nhiều Chiều

In [ ]:
# Ứng dụng PCA trên dữ liệu Wholesale Customers (6 features → 2D)
import pandas as pd

df = pd.read_csv('../Bai thi thu/Course Files/Wholesale customers data.csv')
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X = df[features].values

# Log transform + StandardScaler
X_log = np.log1p(X)
scaler = StandardScaler()
X_scaled_ws = scaler.fit_transform(X_log)

# PCA từ scratch
pca_ws = PCAFromScratch(n_components=2)
X_pca_ws = pca_ws.fit_transform(X_scaled_ws)

print("Wholesale Customers:")
print(f"  Gốc: {X.shape[1]} features")
print(f"  Sau PCA: {X_pca_ws.shape[1]} components")
print(f"  Variance giữ lại: {pca_ws.explained_variance_ratio_[:2].sum()*100:.1f}%")

# Phân tích PCA loading (contribution từng feature vào PC)
print("\n📊 PCA Loadings (đóng góp của từng feature):")
loadings = pd.DataFrame(
    pca_ws.components_,
    columns=features,
    index=['PC1', 'PC2']
)
print(loadings.round(3).to_string())

# Trực quan hóa
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot theo Channel
channel_labels = {1: 'Hotel/Restaurant', 2: 'Retail'}
for ch in [1, 2]:
    mask = df['Channel'] == ch
    axes[0].scatter(X_pca_ws[mask, 0], X_pca_ws[mask, 1], 
                    label=channel_labels[ch], s=40, alpha=0.7)
axes[0].set_xlabel(f'PC1 ({pca_ws.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca_ws.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
axes[0].set_title('PCA: 6D → 2D (theo Channel)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Biplot (vẽ cả loading vectors)
axes[1].scatter(X_pca_ws[:, 0], X_pca_ws[:, 1], c='gray', s=20, alpha=0.3)

# Vẽ loading vectors
scale_factor = 4
for i, feat in enumerate(features):
    axes[1].annotate('', xy=(loadings.iloc[0, i]*scale_factor, loadings.iloc[1, i]*scale_factor),
                     xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='red', lw=2))
    axes[1].text(loadings.iloc[0, i]*scale_factor*1.1, loadings.iloc[1, i]*scale_factor*1.1,
                 feat, fontsize=10, color='red', fontweight='bold')

axes[1].set_xlabel(f'PC1 ({pca_ws.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
axes[1].set_ylabel(f'PC2 ({pca_ws.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
axes[1].set_title('Biplot (Features → PC directions)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 DIỄN GIẢI BIPLOT:")
print("  - PC1 chia rõ Retail (Milk, Grocery, Detergents) vs Hotel/Restaurant (Fresh, Frozen)")
print("  - Milk, Grocery, Detergents_Paper cùng hướng → tương quan mạnh")

---

## 6. Ứng Dụng 2: Giảm Chiều Trước Khi Train Model

In [ ]:
# Demo: PCA giúp tăng tốc model training
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import time

# Sử dụng digits dataset (8x8 images = 64 features)
from sklearn.datasets import load_digits
digits = load_digits()
X_digits = digits.data  # (1797, 64)
y_digits = digits.target

# Chuẩn hóa
scaler = StandardScaler()
X_digits_scaled = scaler.fit_transform(X_digits)

X_train, X_test, y_train, y_test = train_test_split(
    X_digits_scaled, y_digits, test_size=0.2, random_state=42
)

print("=" * 65)
print("SO SÁNH: Training với vs không PCA")
print("=" * 65)

# Không PCA (64 features)
start = time.time()
lr_full = LogisticRegression(max_iter=5000, random_state=42)
lr_full.fit(X_train, y_train)
time_full = time.time() - start
acc_full = accuracy_score(y_test, lr_full.predict(X_test))

# Với PCA (95% variance)
pca_digits = PCAFromScratch(n_components=0.95)
X_train_pca = pca_digits.fit_transform(X_train)
X_test_pca = pca_digits.transform(X_test)

start = time.time()
lr_pca = LogisticRegression(max_iter=5000, random_state=42)
lr_pca.fit(X_train_pca, y_train)
time_pca = time.time() - start
acc_pca = accuracy_score(y_test, lr_pca.predict(X_test_pca))

print(f"\n{'Metric':<25} {'Không PCA':<15} {'Có PCA':<15}")
print("-" * 55)
print(f"{'Số features':<25} {64:<15} {pca_digits.n_components_:<15}")
print(f"{'Accuracy':<25} {acc_full:<15.4f} {acc_pca:<15.4f}")
print(f"{'Thời gian train (s)':<25} {time_full:<15.4f} {time_pca:<15.4f}")
print(f"{'Giảm features':<25} {'---':<15} {(1 - pca_digits.n_components_/64)*100:.0f}%")

print(f"\n✅ PCA giảm {64 - pca_digits.n_components_} features mà accuracy gần như KHÔNG đổi!")

---

## 7. Ứng Dụng 3: Reconstruction & Denoising

PCA có thể **khôi phục dữ liệu** (inverse_transform) và thậm chí **loại bỏ nhiễu**.

In [ ]:
# Demo: PCA cho Denoising ảnh chữ số
from sklearn.datasets import load_digits

digits = load_digits()
X_digits = digits.data

# Thêm nhiễu
np.random.seed(42)
noise = np.random.normal(0, 4, X_digits.shape)
X_noisy = X_digits + noise

# PCA reconstruction (loại bỏ components ít quan trọng = loại nhiễu)
n_components_list = [5, 15, 30, 50]

fig, axes = plt.subplots(len(n_components_list) + 2, 10, figsize=(15, 10))

# Hiển thị 10 ảnh mẫu
for i in range(10):
    # Original
    axes[0, i].imshow(X_digits[i].reshape(8, 8), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Gốc', fontsize=10, rotation=0, labelpad=50)
    
    # Noisy
    axes[1, i].imshow(X_noisy[i].reshape(8, 8), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('+ Nhiễu', fontsize=10, rotation=0, labelpad=50)

# Reconstruction với số components khác nhau
for row, n_comp in enumerate(n_components_list):
    pca_denoise = PCAFromScratch(n_components=n_comp)
    X_transformed = pca_denoise.fit_transform(X_noisy)
    X_reconstructed = pca_denoise.inverse_transform(X_transformed)
    
    for i in range(10):
        axes[row + 2, i].imshow(X_reconstructed[i].reshape(8, 8), cmap='gray')
        axes[row + 2, i].axis('off')
        if i == 0:
            evr = pca_denoise.explained_variance_ratio_[:n_comp].sum()
            axes[row + 2, i].set_ylabel(f'n={n_comp}\n({evr*100:.0f}%)', 
                                         fontsize=9, rotation=0, labelpad=55)

plt.suptitle('PCA Denoising: Loại bỏ nhiễu bằng reconstruction\n(Giữ lại components quan trọng = loại bỏ nhiễu)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 Nhiễu nằm ở components có variance THẤ̊P → PCA loại bỏ tự nhiên!")
print("   15-30 components cho kết quả denoising tốt nhất cho digits dataset")

---

## 8. Bài Thực Hành

### Bài tập: PCA trên dữ liệu Wholesale Customers

**Yêu cầu**:
1. Load và chuẩn hóa dữ liệu (6 features)
2. Áp dụng PCA từ scratch, tìm số components tối ưu (≥ 90% variance)
3. Trực quan hóa 2D và 3D
4. Phân tích loadings: feature nào đóng góp nhiều nhất cho PC1, PC2?
5. So sánh K-Means clustering trên dữ liệu gốc vs dữ liệu PCA

In [ ]:
# BÀI GIẢI MẪU

# 1. Load & chuẩn hóa
df = pd.read_csv('../Bai thi thu/Course Files/Wholesale customers data.csv')
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X = df[features].values
X_log = np.log1p(X)
X_scaled = StandardScaler().fit_transform(X_log)

# 2. PCA - tìm số components
pca_full = PCAFromScratch()
pca_full.fit(X_scaled)

print("Explained Variance Ratio:")
for i, (evr, cum) in enumerate(zip(pca_full.explained_variance_ratio_, pca_full.cumulative_variance_ratio_)):
    bar = '█' * int(evr * 50)
    print(f"  PC{i+1}: {evr*100:5.2f}% | Tích lũy: {cum*100:5.2f}% | {bar}")

# Tìm số components cho 90% variance
pca_90 = PCAFromScratch(n_components=0.90)
pca_90.fit(X_scaled)
print(f"\n→ Cần {pca_90.n_components_} components để đạt 90% variance")

In [ ]:
# 3. Trực quan hóa 2D
pca_2d = PCAFromScratch(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Theo Channel
for ch, name, color in [(1, 'Hotel/Restaurant', '#e74c3c'), (2, 'Retail', '#3498db')]:
    mask = df['Channel'] == ch
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=name, s=40, alpha=0.7)
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA 2D - Theo Channel', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Theo Region
region_names = {1: 'Lisbon', 2: 'Oporto', 3: 'Other'}
for reg, color in [(1, '#e74c3c'), (2, '#3498db'), (3, '#2ecc71')]:
    mask = df['Region'] == reg
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=region_names[reg], s=40, alpha=0.7)
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('PCA 2D - Theo Region', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 4. Phân tích Loadings
loadings = pd.DataFrame(
    pca_2d.components_,
    columns=features,
    index=['PC1', 'PC2']
)

print("PCA Loadings:")
print(loadings.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 6))
loadings.T.plot(kind='barh', ax=ax)
ax.set_xlabel('Loading Value')
ax.set_title('PCA Loadings - Feature Contributions', fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 DIỄN GIẢI:")
print("  PC1: Grocery, Detergents_Paper, Milk đóng góp nhiều nhất → 'Retail purchases'")
print("  PC2: Fresh, Frozen đóng góp nhiều nhất → 'Restaurant/Hotel purchases'")

In [ ]:
# 5. So sánh K-Means trên dữ liệu gốc vs PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# K-Means trên 6D gốc
km_full = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_full = km_full.fit_predict(X_scaled)
sil_full = silhouette_score(X_scaled, labels_full)

# K-Means trên 2D PCA
km_pca = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_pca = km_pca.fit_predict(X_2d)
sil_pca = silhouette_score(X_2d, labels_pca)

print("=" * 50)
print("K-Means: 6D gốc vs 2D PCA")
print("=" * 50)
print(f"  {'Metric':<25} {'6D gốc':<15} {'2D PCA':<15}")
print(f"  {'-'*50}")
print(f"  {'Silhouette Score':<25} {sil_full:<15.4f} {sil_pca:<15.4f}")
print(f"  {'Số features':<25} {6:<15} {2:<15}")

from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(labels_full, labels_pca)
print(f"  {'ARI (tương đồng)':<25} {ari:<15.4f}")

print(f"\n✅ Kết quả clustering trên 2D PCA gần tương đương 6D gốc!")

---

## 📌 TỔNG HỢP: 20% Kiến Thức → 80% Ứng Dụng

| # | Kiến thức cốt lõi | Chi tiết |
|:--|:---|:---|
| 1 | **PCA = Xoay trục → Giữ hướng có variance lớn nhất** | Không tạo features mới, chỉ TỔ HỢP TUYẾN TÍNH features cũ |
| 2 | **LUÔN chuẩn hóa trước PCA** | Không chuẩn hóa → feature có scale lớn chi phối PC1 |
| 3 | **Chọn n_components bằng Cumulative Variance ≥ 95%** | `pca = PCA(n_components=0.95)` — đây là cách phổ biến nhất |
| 4 | **PCA Loadings → Diễn giải PC** | Loading lớn = feature đóng góp nhiều cho PC đó |
| 5 | **3 ứng dụng chính** | ① Trực quan hóa, ② Giảm chiều trước training, ③ Denoising |

### Code Template nhanh:

```python
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Chuẩn hóa
X_scaled = StandardScaler().fit_transform(X)

# 2. PCA - giữ 95% variance
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)
print(f'Giảm từ {X.shape[1]}D → {X_pca.shape[1]}D')
print(f'Variance giữ lại: {pca.explained_variance_ratio_.sum()*100:.1f}%')

# 3. PCA cho visualization
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)
plt.scatter(X_2d[:, 0], X_2d[:, 1])
```

### ⚠️ Khi nào KHÔNG dùng PCA:
- Dữ liệu có quan hệ **phi tuyến** → dùng t-SNE, UMAP
- Cần **diễn giải features** → PCA biến đổi features nên khó giải thích
- Dữ liệu **ít features** (< 10) → thường không cần giảm chiều

---

**→ Bài tiếp theo: [Bài 4] ICA - Independent Component Analysis**